In [ ]:
from gaussed.gp import GP # type: ignore
from gaussed.gp.kernels import MaternKernel, MaternParams # type: ignore
from gaussed.gp.means import ZeroMean # type: ignore
from gaussed.engines.gp_backends import CholeskyBackend  # type: ignore
from gaussed.gp.gp_ops import Eval, Grad, Stack # type: ignore
from gaussed.utils.constraints import Positive # type: ignore

# shared hyperparameters (can be optimised or MCMC’d)
theta = MaternParams(raw_ell=0.0, raw_sigma2=0.0, nu=2.5)
k = MaternKernel(theta)  # kernel references params (shared across GPs)

gp = GP(domain, codomain, kernel=k, mean=ZeroMean(), noise_raw=-3.0)  # type: ignore # raw noise; transform inside

# Observations: values at X plus, say, x-derivative at Xd
obs = Stack([
    Eval(X),                # f(X) # type: ignore
    Grad(Xd, axis=0),       # ∂f/∂x₀ (Xd) # type: ignore
])
y = jnp.concatenate([y_val, y_grad])          # type: ignore[reportUndefinedVariable] # shape (n_obs,)
noise = DiagonalNoise(sigma2=jnp.concatenate([σ2_val, σ2_grad])) # type: ignore[reportUndefinedVariable]

post = gp.condition(obs, y, noise, backend=CholeskyBackend())

# Predictions anywhere, from any probes (including mixed blocks)
Q = Stack([Eval(Xtest), Grad(Xtest, axis=1)]) # type: ignore[reportUndefinedVariable]
m = post.mean(Q)                               # (n_Q,)
S = post.cov(Q, Q)                              # (n_Q, n_Q)
v = post.variance(Eval(Xtest))                  # type: ignore[reportUndefinedVariable] # (n_test,)
samples = post.sample(key, Eval(Xtest), n=10)   # type: ignore[reportUndefinedVariable] # (10, n_test)

# Cross-covariances between different probe types (e.g., value vs gradient)
K_val_grad = post.cross_cov(Eval(Xtest), Grad(Xtest, axis=0)) # type: ignore[reportUndefinedVariable]
